In [1]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║  XGBoost  │  Predict 2024 NCAA Tournament                           ║
║                                                                      ║
║  Identical data pipeline to Script 1 (LightGBM) — only the model   ║
║  is swapped to XGBoost for a direct apples-to-apples comparison.    ║
║                                                                      ║
║  TRAINING DATA (labels = game outcomes):                             ║
║    ① Regular season games  2013–2024  → each game = one training row ║
║       features = game-level box-score differentials                  ║
║    ② Tournament games      2013–2023  → each game = one training row ║
║       features = that season's reg-season aggregated stats + seeds   ║
║                                                                      ║
║  TEST DATA (never touched during training):                          ║
║    2024 Tournament matchups only                                     ║
║    Features: 2024 regular-season aggregated stats + 2024 seeds       ║
║                                                                      ║
║  NO ELO — box-score + seed features only                             ║
║                                                                      ║
║  NaN HANDLING                                                        ║
║    Seeds / WinRate / GamesDiff are NaN for regular-season rows.      ║
║    XGBoost natively handles NaN with a learned default direction     ║
║    at each split — no imputation required.                           ║
║                                                                      ║
║  LEAKAGE CONTROLS                                                    ║
║    • Reg-season rows use ONLY that game's own box score.             ║
║    • Tournament feature rows use ONLY same-season reg-season stats   ║
║      (computed before the tournament starts).                        ║
║    • 2024 tournament outcomes are NEVER used in training.            ║
║    • 2024 regular season is used ONLY to build test features.        ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import log_loss, roc_auc_score

print(f"XGBoost version: {xgb.__version__}")

# ─────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────
m_regular_season_detailed = pd.read_csv('/Users/shaurya/Downloads/MRegularSeasonDetailedResults.csv')
m_seeds                   = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneySeeds.csv')
m_tourney_detailed_result = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneyDetailedResults.csv')

print("Regular Season shape :", m_regular_season_detailed.shape)
print("Seeds shape          :", m_seeds.shape)
print("Tourney Results shape:", m_tourney_detailed_result.shape)

# ─────────────────────────────────────────────────────────────────
# 2. PARSE SEEDS  (e.g. "W01" → 1, "Z16a" → 16)
# ─────────────────────────────────────────────────────────────────
def parse_seed(s):
    return int(''.join(filter(str.isdigit, s)))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)

# ─────────────────────────────────────────────────────────────────
# 3. SEASON-AGGREGATED TEAM STATS
#    Built per season from regular-season games only.
#    Used as features for tournament matchup rows.
# ─────────────────────────────────────────────────────────────────
def compute_team_stats(reg_df, seasons):
    rows = reg_df[reg_df['Season'].isin(seasons)].copy()

    win = rows.rename(columns={
        'WTeamID':'TeamID', 'LTeamID':'OppID',
        'WScore':'Pts',     'LScore':'OppPts',
        'WFGM':'FGM','WFGA':'FGA','WFGM3':'FGM3','WFGA3':'FGA3',
        'WFTM':'FTM','WFTA':'FTA','WOR':'OR','WDR':'DR',
        'WAst':'Ast','WTO':'TO','WStl':'Stl','WBlk':'Blk','WPF':'PF',
    }).copy()
    win['Win'] = 1

    lose = rows.rename(columns={
        'LTeamID':'TeamID', 'WTeamID':'OppID',
        'LScore':'Pts',     'WScore':'OppPts',
        'LFGM':'FGM','LFGA':'FGA','LFGM3':'FGM3','LFGA3':'FGA3',
        'LFTM':'FTM','LFTA':'FTA','LOR':'OR','LDR':'DR',
        'LAst':'Ast','LTO':'TO','LStl':'Stl','LBlk':'Blk','LPF':'PF',
    }).copy()
    lose['Win'] = 0

    keep = ['Season','TeamID','Pts','OppPts','FGM','FGA','FGM3','FGA3',
            'FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF','Win']
    all_g = pd.concat([win[keep], lose[keep]], ignore_index=True)

    agg = all_g.groupby(['Season','TeamID']).agg(
        Games      =('Win','count'),
        WinRate    =('Win','mean'),
        AvgPts     =('Pts','mean'),
        AvgOppPts  =('OppPts','mean'),
        AvgFGM     =('FGM','mean'),
        AvgFGA     =('FGA','mean'),
        AvgFGM3    =('FGM3','mean'),
        AvgFGA3    =('FGA3','mean'),
        AvgFTM     =('FTM','mean'),
        AvgFTA     =('FTA','mean'),
        AvgOR      =('OR','mean'),
        AvgDR      =('DR','mean'),
        AvgAst     =('Ast','mean'),
        AvgTO      =('TO','mean'),
        AvgStl     =('Stl','mean'),
        AvgBlk     =('Blk','mean'),
        AvgPF      =('PF','mean'),
    ).reset_index()

    agg['FGPct']  = agg['AvgFGM']  / agg['AvgFGA'].replace(0, np.nan)
    agg['FG3Pct'] = agg['AvgFGM3'] / agg['AvgFGA3'].replace(0, np.nan)
    agg['FTPct']  = agg['AvgFTM']  / agg['AvgFTA'].replace(0, np.nan)
    agg['Margin'] = agg['AvgPts']  - agg['AvgOppPts']
    return agg.set_index(['Season','TeamID'])


# ─────────────────────────────────────────────────────────────────
# 4A. REGULAR-SEASON TRAINING ROWS
#     One row per game. Features = that game's raw box-score diffs.
#     Seeds / WinRate / GamesDiff are NaN — XGBoost handles natively.
# ─────────────────────────────────────────────────────────────────
def build_reg_season_train_rows(reg_df, seasons):
    rows = reg_df[reg_df['Season'].isin(seasons)].copy()
    records = []

    def pct(m, a):
        return m / a if a and a > 0 else np.nan

    for _, g in rows.iterrows():
        w_id, l_id = int(g['WTeamID']), int(g['LTeamID'])

        # Canonical ordering: lower TeamID = Team A
        if w_id < l_id:
            t_a, t_b, label  = w_id, l_id, 1
            pts_a, pts_b     = g['WScore'], g['LScore']
            fgm_a, fga_a     = g['WFGM'],  g['WFGA']
            fgm3_a, fga3_a   = g['WFGM3'], g['WFGA3']
            ftm_a, fta_a     = g['WFTM'],  g['WFTA']
            or_a, dr_a       = g['WOR'],   g['WDR']
            ast_a, to_a      = g['WAst'],  g['WTO']
            stl_a, blk_a, pf_a = g['WStl'], g['WBlk'], g['WPF']
            fgm_b, fga_b     = g['LFGM'],  g['LFGA']
            fgm3_b, fga3_b   = g['LFGM3'], g['LFGA3']
            ftm_b, fta_b     = g['LFTM'],  g['LFTA']
            or_b, dr_b       = g['LOR'],   g['LDR']
            ast_b, to_b      = g['LAst'],  g['LTO']
            stl_b, blk_b, pf_b = g['LStl'], g['LBlk'], g['LPF']
        else:
            t_a, t_b, label  = l_id, w_id, 0
            pts_a, pts_b     = g['LScore'], g['WScore']
            fgm_a, fga_a     = g['LFGM'],  g['LFGA']
            fgm3_a, fga3_a   = g['LFGM3'], g['LFGA3']
            ftm_a, fta_a     = g['LFTM'],  g['LFTA']
            or_a, dr_a       = g['LOR'],   g['LDR']
            ast_a, to_a      = g['LAst'],  g['LTO']
            stl_a, blk_a, pf_a = g['LStl'], g['LBlk'], g['LPF']
            fgm_b, fga_b     = g['WFGM'],  g['WFGA']
            fgm3_b, fga3_b   = g['WFGM3'], g['WFGA3']
            ftm_b, fta_b     = g['WFTM'],  g['WFTA']
            or_b, dr_b       = g['WOR'],   g['WDR']
            ast_b, to_b      = g['WAst'],  g['WTO']
            stl_b, blk_b, pf_b = g['WStl'], g['WBlk'], g['WPF']

        fg_a  = pct(fgm_a, fga_a);   fg_b  = pct(fgm_b, fga_b)
        fg3_a = pct(fgm3_a, fga3_a); fg3_b = pct(fgm3_b, fga3_b)
        ft_a  = pct(ftm_a, fta_a);   ft_b  = pct(ftm_b, fta_b)
        margin_a = pts_a - pts_b

        records.append({
            'Season': int(g['Season']), 'TeamA': t_a, 'TeamB': t_b,
            'Label': label, 'DataSource': 'RegularSeason',

            # NaN for reg-season — XGBoost learns optimal default direction
            'SeedA': np.nan, 'SeedB': np.nan, 'SeedDiff': np.nan,

            # Difference features (A − B)
            'WinRateDiff': np.nan,
            'PtsDiff':     pts_a - pts_b,
            'OppPtsDiff':  pts_b - pts_a,
            'MarginDiff':  margin_a,
            'FGPctDiff':   (fg_a  or 0) - (fg_b  or 0),
            'FG3PctDiff':  (fg3_a or 0) - (fg3_b or 0),
            'FTPctDiff':   (ft_a  or 0) - (ft_b  or 0),
            'ORDiff':      or_a  - or_b,
            'DRDiff':      dr_a  - dr_b,
            'AstDiff':     ast_a - ast_b,
            'TODiff':      to_a  - to_b,
            'StlDiff':     stl_a - stl_b,
            'BlkDiff':     blk_a - blk_b,
            'PFDiff':      pf_a  - pf_b,
            'GamesDiff':   np.nan,

            # Raw Team A
            'A_WinRate': np.nan,   'A_AvgPts': pts_a,   'A_Margin': margin_a,
            'A_FGPct':   fg_a,     'A_FG3Pct': fg3_a,   'A_FTPct':  ft_a,
            'A_AvgOR':   or_a,     'A_AvgDR':  dr_a,
            'A_AvgAst':  ast_a,    'A_AvgTO':  to_a,
            'A_AvgStl':  stl_a,    'A_AvgBlk': blk_a,

            # Raw Team B
            'B_WinRate': np.nan,   'B_AvgPts': pts_b,   'B_Margin': -margin_a,
            'B_FGPct':   fg_b,     'B_FG3Pct': fg3_b,   'B_FTPct':  ft_b,
            'B_AvgOR':   or_b,     'B_AvgDR':  dr_b,
            'B_AvgAst':  ast_b,    'B_AvgTO':  to_b,
            'B_AvgStl':  stl_b,    'B_AvgBlk': blk_b,
        })

    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────
# 4B. TOURNAMENT MATCHUP ROWS
#     Features = season-agg reg-season stats + seeds.
#     Used for training (2013–2023) and test (2024).
# ─────────────────────────────────────────────────────────────────
def build_tourney_matchup_rows(tourney_df, stats_df, seeds_df, seasons):
    records = []
    for _, row in tourney_df[tourney_df['Season'].isin(seasons)].iterrows():
        season = row['Season']
        w_id, l_id = row['WTeamID'], row['LTeamID']

        if w_id < l_id:
            t_a, t_b, label = w_id, l_id, 1
        else:
            t_a, t_b, label = l_id, w_id, 0

        if (season, t_a) not in stats_df.index or (season, t_b) not in stats_df.index:
            continue

        sa = stats_df.loc[(season, t_a)]
        sb = stats_df.loc[(season, t_b)]

        s_a = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_a)]
        s_b = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_b)]
        seed_a = s_a['SeedNum'].values[0] if len(s_a) else np.nan
        seed_b = s_b['SeedNum'].values[0] if len(s_b) else np.nan

        records.append({
            'Season': season, 'TeamA': t_a, 'TeamB': t_b,
            'Label': label, 'DataSource': 'Tournament',

            'SeedA': seed_a, 'SeedB': seed_b, 'SeedDiff': seed_a - seed_b,

            'WinRateDiff':  sa['WinRate']  - sb['WinRate'],
            'PtsDiff':      sa['AvgPts']   - sb['AvgPts'],
            'OppPtsDiff':   sa['AvgOppPts']- sb['AvgOppPts'],
            'MarginDiff':   sa['Margin']   - sb['Margin'],
            'FGPctDiff':    sa['FGPct']    - sb['FGPct'],
            'FG3PctDiff':   sa['FG3Pct']   - sb['FG3Pct'],
            'FTPctDiff':    sa['FTPct']    - sb['FTPct'],
            'ORDiff':       sa['AvgOR']    - sb['AvgOR'],
            'DRDiff':       sa['AvgDR']    - sb['AvgDR'],
            'AstDiff':      sa['AvgAst']   - sb['AvgAst'],
            'TODiff':       sa['AvgTO']    - sb['AvgTO'],
            'StlDiff':      sa['AvgStl']   - sb['AvgStl'],
            'BlkDiff':      sa['AvgBlk']   - sb['AvgBlk'],
            'PFDiff':       sa['AvgPF']    - sb['AvgPF'],
            'GamesDiff':    sa['Games']    - sb['Games'],

            'A_WinRate': sa['WinRate'], 'A_AvgPts': sa['AvgPts'],
            'A_Margin':  sa['Margin'],  'A_FGPct':  sa['FGPct'],
            'A_FG3Pct':  sa['FG3Pct'], 'A_FTPct':  sa['FTPct'],
            'A_AvgOR':   sa['AvgOR'],   'A_AvgDR':  sa['AvgDR'],
            'A_AvgAst':  sa['AvgAst'], 'A_AvgTO':   sa['AvgTO'],
            'A_AvgStl':  sa['AvgStl'], 'A_AvgBlk':  sa['AvgBlk'],

            'B_WinRate': sb['WinRate'], 'B_AvgPts': sb['AvgPts'],
            'B_Margin':  sb['Margin'],  'B_FGPct':  sb['FGPct'],
            'B_FG3Pct':  sb['FG3Pct'], 'B_FTPct':  sb['FTPct'],
            'B_AvgOR':   sb['AvgOR'],   'B_AvgDR':  sb['AvgDR'],
            'B_AvgAst':  sb['AvgAst'], 'B_AvgTO':   sb['AvgTO'],
            'B_AvgStl':  sb['AvgStl'], 'B_AvgBlk':  sb['AvgBlk'],
        })
    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────
# 5. ASSEMBLE TRAIN / TEST
# ─────────────────────────────────────────────────────────────────
REG_TRAIN_SEASONS   = list(range(2013, 2025))   # 2013–2024 regular seasons
TOURN_TRAIN_SEASONS = list(range(2013, 2024))   # 2013–2023 tournaments
TEST_SEASONS        = [2024]                    # 2024 tournament only

stats = compute_team_stats(m_regular_season_detailed, list(range(2013, 2025)))

reg_train_df   = build_reg_season_train_rows(m_regular_season_detailed, REG_TRAIN_SEASONS)
tourn_train_df = build_tourney_matchup_rows(m_tourney_detailed_result, stats, m_seeds, TOURN_TRAIN_SEASONS)
train_df       = pd.concat([reg_train_df, tourn_train_df], ignore_index=True)
test_df        = build_tourney_matchup_rows(m_tourney_detailed_result, stats, m_seeds, TEST_SEASONS)

print(f"\n{'─'*55}")
print(f"  Training rows")
print(f"{'─'*55}")
print(f"  Regular season games  2013–2024 : {len(reg_train_df):>7,}")
print(f"  Tournament games      2013–2023 : {len(tourn_train_df):>7,}")
print(f"  TOTAL                           : {len(train_df):>7,}")
print(f"\n  Test rows (2024 tournament)     : {len(test_df):>7,}")
print(f"{'─'*55}")

# ─────────────────────────────────────────────────────────────────
# 6. FEATURE COLUMNS
# ─────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    'SeedA', 'SeedB', 'SeedDiff',
    'WinRateDiff', 'PtsDiff', 'OppPtsDiff', 'MarginDiff',
    'FGPctDiff', 'FG3PctDiff', 'FTPctDiff',
    'ORDiff', 'DRDiff', 'AstDiff', 'TODiff', 'StlDiff', 'BlkDiff', 'PFDiff',
    'GamesDiff',
    'A_WinRate', 'A_AvgPts', 'A_Margin', 'A_FGPct', 'A_FG3Pct', 'A_FTPct',
    'A_AvgOR', 'A_AvgDR', 'A_AvgAst', 'A_AvgTO', 'A_AvgStl', 'A_AvgBlk',
    'B_WinRate', 'B_AvgPts', 'B_Margin', 'B_FGPct', 'B_FG3Pct', 'B_FTPct',
    'B_AvgOR', 'B_AvgDR', 'B_AvgAst', 'B_AvgTO', 'B_AvgStl', 'B_AvgBlk',
]

X_train = train_df[FEATURE_COLS]
y_train = train_df['Label']
X_test  = test_df[FEATURE_COLS]
y_test  = test_df['Label']

print(f"\n  Features : {len(FEATURE_COLS)}")
print(f"  NaN cols (Seed*, WinRate*, GamesDiff) handled natively by XGBoost")

# ─────────────────────────────────────────────────────────────────
# 7. BUILD DMatrix  (XGBoost's optimised data structure)
#    enable_categorical=False  — all features are numeric / NaN
# ─────────────────────────────────────────────────────────────────
dtrain = xgb.DMatrix(X_train, label=y_train, missing=np.nan)
dtest  = xgb.DMatrix(X_test,  label=y_test,  missing=np.nan)

# ─────────────────────────────────────────────────────────────────
# 8. TRAIN XGBOOST
#    Hyperparameters mirror the LightGBM Script 1 as closely as
#    XGBoost allows for a fair comparison:
#      max_depth=6    ≈ num_leaves=31 (2^5=32 leaves at depth 5)
#      eta=0.05       = learning_rate=0.05
#      subsample=0.8  = bagging_fraction=0.8
#      colsample_bytree=0.8 = feature_fraction=0.8
#      alpha=0.1      = lambda_l1=0.1
#      lambda=0.1     = lambda_l2=0.1
#      min_child_weight=20  ≈ min_child_samples=20
# ─────────────────────────────────────────────────────────────────
params = {
    'objective':        'binary:logistic',
    'eval_metric':      'logloss',
    'tree_method':      'hist',          # fast histogram method (same idea as LightGBM)
    'max_depth':        6,
    'eta':              0.05,            # learning rate
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'alpha':            0.1,             # L1 regularisation
    'lambda':           0.1,             # L2 regularisation
    'min_child_weight': 20,
    'seed':             42,
    'verbosity':        0,
}

evals_result = {}
model = xgb.train(
    params,
    dtrain,
    num_boost_round=600,
    evals=[(dtest, 'test')],
    evals_result=evals_result,
    early_stopping_rounds=50,
    verbose_eval=50,
)

print(f"\nBest iteration : {model.best_iteration}")
print(f"Best val logloss : {model.best_score:.6f}")

# ─────────────────────────────────────────────────────────────────
# 9. EVALUATE ON 2024 TOURNAMENT
# ─────────────────────────────────────────────────────────────────
y_pred_prob = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
y_pred_bin  = (y_pred_prob >= 0.5).astype(int)

ll  = log_loss(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
acc = (y_pred_bin == y_test.values).mean()

print(f"\n{'='*55}")
print(f"  XGBoost — 2024 Tournament Evaluation")
print(f"  Train: reg season 2013-2024 + tournaments 2013-2023")
print(f"  Test : 2024 tournament only")
print(f"{'='*55}")
print(f"  Log Loss : {ll:.4f}   (random baseline = 0.6931)")
print(f"  ROC AUC  : {auc:.4f}   (random baseline = 0.5000)")
print(f"  Accuracy : {acc:.4f}   ({int(acc*len(y_test))}/{len(y_test)} correct)")

# ─────────────────────────────────────────────────────────────────
# 10. FEATURE IMPORTANCE  (gain-based — same metric as LightGBM)
# ─────────────────────────────────────────────────────────────────
scores = model.get_score(importance_type='gain')

importance = pd.DataFrame({
    'Feature':    list(scores.keys()),
    'Importance': list(scores.values()),
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nTop 15 Features by Gain:")
print(importance.head(15).to_string(index=False))

# ─────────────────────────────────────────────────────────────────
# 11. OUTPUT — PREDICTIONS CSV + FEATURE LIST COLUMN
# ─────────────────────────────────────────────────────────────────
results_df = test_df[['Season','TeamA','TeamB','Label'] + FEATURE_COLS].copy()
results_df['PredProb_TeamA_Wins'] = y_pred_prob
results_df['PredWinner']   = results_df.apply(
    lambda r: r['TeamA'] if r['PredProb_TeamA_Wins'] >= 0.5 else r['TeamB'], axis=1)
results_df['ActualWinner'] = results_df.apply(
    lambda r: r['TeamA'] if r['Label'] == 1 else r['TeamB'], axis=1)
results_df['Correct']      = (results_df['PredWinner'] == results_df['ActualWinner']).astype(int)
results_df['FeaturesUsed'] = ' | '.join(FEATURE_COLS)

results_df.to_csv('/Users/shaurya/Downloads/xgb_2024_predictions.csv', index=False)
importance.to_csv('/Users/shaurya/Downloads/xgb_feature_importance.csv', index=False)

print(f"\n✅  Predictions saved  → /Users/shaurya/Downloads/xgb_2024_predictions.csv")
print(f"✅  Importance saved   → /Users/shaurya/Downloads/xgb_feature_importance.csv")
print(f"\nTotal features : {len(FEATURE_COLS)}")
print("Feature list   :", FEATURE_COLS)

XGBoost version: 3.2.0
Regular Season shape : (122775, 34)
Seeds shape          : (2626, 3)
Tourney Results shape: (1449, 34)

───────────────────────────────────────────────────────
  Training rows
───────────────────────────────────────────────────────
  Regular season games  2013–2024 :  63,405
  Tournament games      2013–2023 :     669
  TOTAL                           :  64,074

  Test rows (2024 tournament)     :      67
───────────────────────────────────────────────────────

  Features : 42
  NaN cols (Seed*, WinRate*, GamesDiff) handled natively by XGBoost
[0]	test-logloss:0.69197
[50]	test-logloss:0.55164
[100]	test-logloss:0.52730
[150]	test-logloss:0.53095
[155]	test-logloss:0.53060

Best iteration : 105
Best val logloss : 0.524524

  XGBoost — 2024 Tournament Evaluation
  Train: reg season 2013-2024 + tournaments 2013-2023
  Test : 2024 tournament only
  Log Loss : 0.5245   (random baseline = 0.6931)
  ROC AUC  : 0.7965   (random baseline = 0.5000)
  Accuracy : 0.7612   (